# Rules you can act on: rules of the tree, rulesets and a churn campaign

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daviddiazsolis/c50py/blob/main/examples/notebooks/04_rules_rulesets_churn_campaigns.ipynb)

A gradient boosting model tells you **who** is likely to leave. A decision tree also tells you
**why**: every prediction comes from a rule, a short list of conditions on the customer's data.
That turns a churn model into a segmentation you can act on. Customers who leave for the same
reason can receive the same retention campaign, and each customer at risk can be traced to the
rule that flagged them.

`c50py` gives two families of rules, and it is worth keeping them apart:

| | **Rules of the tree** | **Ruleset** (C5.0's `rules` mode) |
|---|---|---|
| Model | `C5Classifier` | `C5RulesClassifier`, or `tree.build_ruleset(X, y)` |
| One rule per | leaf of the tree | rule kept after simplification and selection |
| Overlap | never: each customer follows exactly one rule | possible: a customer can satisfy several |
| List them | `tree.export_rules()` | `ruleset.export_ruleset()` |
| Which rule for each customer | `tree.apply_rules(X)` (`tree.apply(X)` gives the number) | `ruleset.apply_ruleset(X)` |
| Predict with | `tree.predict(X)` | `ruleset.predict(X)` |

The rules of the tree are the tree written as text: faithful, exhaustive, and often long, because
every rule repeats the tests of the path from the root. A ruleset is a different, smaller model
built from them, following C5.0's `rules` mode (a refinement of Quinlan's C4.5rules, 1993,
chapter 5): tests that do not help are dropped and redundant rules are discarded. This notebook builds both on a telecom churn
dataset and turns the ruleset into a campaign plan.

In [1]:
# Colab / a fresh environment: install c50py (0.5.2 or later)
import importlib, subprocess, sys
try:
    import c50py
    assert tuple(int(v) for v in c50py.__version__.split(".")[:3]) >= (0, 5, 2)
except (ImportError, AssertionError):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "c50py>=0.5.2"], check=True)
    import c50py
    importlib.reload(c50py)
print("c50py", c50py.__version__)

c50py 0.5.2


## 1. The data: 5,000 telecom customers

The `churn` dataset on OpenML (id 40701) has one row per customer of a US telecom operator: the
plans they have (international, voicemail), how many minutes they talk by day, evening, night and
internationally, how many times they called customer service, and whether they left. The charge
columns are the minutes times a fixed price, so we drop them, together with the phone number.

Nothing else is needed: no one-hot encoding, no scaling. The plan columns are categories and
`c50py` uses them as such.

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from c50py import C5Classifier, C5RulesClassifier

d = fetch_openml(data_id=40701, as_frame=True, parser="pandas")
X = d.data.drop(columns=["phone_number", "total_day_charge", "total_eve_charge",
                         "total_night_charge", "total_intl_charge"])
X = X.rename(columns={"number_customer_service_calls": "service_calls",
                      "number_vmail_messages": "vmail_messages"})
X["state"] = X["state"].astype("category")
X["area_code"] = X["area_code"].astype(str).astype("category")
X["service_calls"] = X["service_calls"].astype(int)
for c in ["international_plan", "voice_mail_plan"]:
    X[c] = X[c].astype(str).map({"0": "no", "1": "yes"}).astype("category")
y = np.where(d.target.astype(int) == 1, "churn", "stays")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(X.shape, " churn rate:", round((y == "churn").mean(), 3))
X.head()

(5000, 15)  churn rate: 0.141


,state,account_length,area_code,international_plan,voice_mail_plan,vmail_messages,total_day_minutes,total_day_calls,total_eve_minutes,total_eve_calls,total_night_minutes,total_night_calls,total_intl_minutes,total_intl_calls,service_calls
0,16,128,415,no,yes,25,265.1,110,197.4,99,244.7,91,10.0,3,1
1,35,107,415,no,yes,26,161.6,123,195.5,103,254.4,103,13.7,3,1
2,31,137,415,no,no,0,243.4,114,121.2,110,162.6,104,12.2,5,0
3,35,84,408,yes,no,0,299.4,71,61.9,88,196.9,89,6.6,7,2
4,36,75,415,yes,no,0,166.7,113,148.3,122,186.9,121,10.1,3,3


## 2. The tree and its rules

A `C5Classifier` with C5.0's default settings. Every leaf of the tree is a rule: the conjunction of
the tests on the path from the root to the leaf. `export_rules()` lists them.

In [3]:
tree = C5Classifier().fit(X_train, y_train)
rules = tree.export_rules()
print(f"{len(rules)} rules of the tree (leaves); test accuracy {tree.score(X_test, y_test):.3f}\n")
for r in rules[:6]:
    print(r)

39 rules of the tree (leaves); test accuracy 0.954

total_day_minutes <= 254.0000 AND service_calls <= 3.5000 AND international_plan IN {no} AND total_day_minutes <= 220.7500 => stays
total_day_minutes <= 254.0000 AND service_calls <= 3.5000 AND international_plan IN {no} AND total_day_minutes > 220.7500 AND total_eve_minutes <= 242.8500 => stays
total_day_minutes <= 254.0000 AND service_calls <= 3.5000 AND international_plan IN {no} AND total_day_minutes > 220.7500 AND total_eve_minutes > 242.8500 AND voice_mail_plan IN {no} AND total_night_minutes <= 223.9500 AND total_eve_minutes <= 259.3000 => stays
total_day_minutes <= 254.0000 AND service_calls <= 3.5000 AND international_plan IN {no} AND total_day_minutes > 220.7500 AND total_eve_minutes > 242.8500 AND voice_mail_plan IN {no} AND total_night_minutes <= 223.9500 AND total_eve_minutes > 259.3000 AND total_day_minutes <= 233.3500 AND total_eve_minutes <= 272.0500 => stays
total_day_minutes <= 254.0000 AND service_calls <= 3.5000 AN

Because the rules of the tree are mutually exclusive, every customer follows exactly one of them.
`apply_rules(X)` returns that rule for each row, with its number (the position in
`export_rules()`, the same number that scikit-learn's `apply` would give) and the prediction.

In [4]:
tree.apply_rules(X_test.head(5))

,rule_id,rule,prediction
3920,22,total_day_minutes <= 254.0000 AND service_call...,stays
4793,0,total_day_minutes <= 254.0000 AND service_call...,stays
4607,0,total_day_minutes <= 254.0000 AND service_call...,stays
2814,0,total_day_minutes <= 254.0000 AND service_call...,stays
728,38,total_day_minutes > 254.0000 AND voice_mail_pl...,stays


This is already useful: for any customer you can say why the model flags them. The drawback is
length. Look at the churn rules of the tree, sorted by length: each one carries all the tests of
its path, including tests that only mattered to separate other customers higher up in the tree.

In [5]:
churn_rules = sorted([r for r in rules if r.endswith("=> churn")], key=lambda r: -r.count(" AND "))
print(f"{len(churn_rules)} churn rules of the tree; the longest has {churn_rules[0].count(' AND ') + 1} tests:\n")
print(churn_rules[0])

19 churn rules of the tree; the longest has 10 tests:

total_day_minutes <= 254.0000 AND service_calls <= 3.5000 AND international_plan IN {no} AND total_day_minutes > 220.7500 AND total_eve_minutes > 242.8500 AND voice_mail_plan IN {no} AND total_night_minutes <= 223.9500 AND total_eve_minutes > 259.3000 AND total_day_minutes <= 233.3500 AND total_eve_minutes > 272.0500 => churn


## 3. From rules of the tree to a ruleset

C5.0 builds a ruleset in four steps:

1. **Candidates.** The path from the root to every node of the tree (not only the leaves) is a
   candidate rule, predicting the majority class of the node. Tests on the same column are merged:
   `total_day_minutes <= 254 AND total_day_minutes <= 220.75` becomes `total_day_minutes <= 220.75`.
2. **Simplify.** Tests are dropped one at a time, starting with the one whose removal leaves the
   most reliable rule, until every further removal would make the rule less reliable. A rule that
   says too little about the class for its length, or that is barely better than the share of its
   class in the data, is discarded.
3. **Select.** A subset of the candidates is kept that describes the training data with the fewest
   bits (the *minimum description length* principle, MDL): enough rules to classify the customers
   well, without rules that add nothing.
4. **Default.** Customers that no rule covers get a default class.

**How reliable is a rule? The Laplace estimate.** A rule covers $N$ training customers and gets
$E$ of them wrong. The observed error rate $E/N$ is optimistic for small rules (a rule with 2
customers and no errors is not "100% right"). C5.0 uses Laplace's *rule of succession* instead:

$$\hat{e} = \frac{E + 1}{N + 2}$$

- $N$: number of training customers that satisfy all the tests of the rule (weighted, if there are case weights).
- $E$: how many of those belong to a class other than the rule's (its errors on the training data).
- $\hat{e}$: the estimated error rate of the rule on new customers.

In words: *pretend you saw one more right and one more wrong case.* For large rules it hardly
changes anything; for small ones it pulls the error rate towards 1/2. A rule with 3 customers
and no errors gets $\hat{e} = 0.20$, one with 55 customers and no errors gets $0.018$. Its
complement, $1 - \hat{e} = (N - E + 1)/(N + 2)$, is the **confidence** reported for each rule.

**Is a rule worth stating? Description length.** Step 2 keeps a test only if the rule needs it,
and step 3 keeps a rule only if it pays for itself. Both use the same currency, bits. The cost of
a ruleset of $R$ rules that makes $E$ errors on $N$ training customers with $K$ classes is

$$L = 0.23\,\Big(\textstyle\sum_{r} \text{bits}(r) - \log_2 R!\Big) + E\,b_{\text{err}} + (N - E)\,b_{\text{ok}} + E\,\log_2(K - 1)$$

- $\text{bits}(r)$: the bits to state rule $r$: for each test, which column it uses (about $\log_2$
  of the number of columns) plus which threshold or which categories.
- $\log_2 R!$: the order in which the rules are listed does not matter, so that information is
  not charged.
- $0.23$: C5.0's discount on the cost of the rules (the stated costs overestimate what a rule
  really needs; the value was tuned by Quinlan).
- $b_{\text{err}} = -\log_2 r$ and $b_{\text{ok}} = -\log_2(1 - r)$: the bits to flag a customer as
  misclassified or correctly classified, where $r$ is the expected error rate (the training error
  rate of the tree, slightly smoothed).
- $E \log_2(K - 1)$: the bits to say which class is right for each error (zero with two classes).

In words: *a rule is kept if the errors it fixes are worth more than the bits it takes to write
it down.* A case covered by several rules is classified by their votes (each rule votes for its
class with its confidence); a case no rule covers counts as an error. The cell below gives the
magnitudes for this data.

In [6]:
def laplace_error(E, N):
    return (E + 1) / (N + 2)

for E, N in [(0, 3), (0, 55), (1, 103), (5, 81)]:
    print(f"E = {E:3d}, N = {N:4d}:  observed error {E / N:.3f}   Laplace estimate {laplace_error(E, N):.3f}")

r = (tree.predict(X_train) != y_train).mean()       # training error rate of the tree
b_err, b_ok = -np.log2(r), -np.log2(1 - r)
n_columns = X.shape[1]
print(f"\nexpected error rate r = {r:.3f}: flagging an error costs {b_err:.2f} bits, a correct case {b_ok:.3f} bits,")
print(f"so each error a rule removes saves about {b_err - b_ok:.1f} bits; naming one of {n_columns} columns costs "
      f"{np.log2(n_columns):.1f} bits,")
print(f"so a rule with three tests (about {3 * (np.log2(n_columns) + 4):.0f} bits, discounted to "
      f"{0.23 * 3 * (np.log2(n_columns) + 4):.0f}) pays for itself if it removes two or three errors.")

E =   0, N =    3:  observed error 0.000   Laplace estimate 0.200
E =   0, N =   55:  observed error 0.000   Laplace estimate 0.018
E =   1, N =  103:  observed error 0.010   Laplace estimate 0.019
E =   5, N =   81:  observed error 0.062   Laplace estimate 0.072

expected error rate r = 0.033: flagging an error costs 4.90 bits, a correct case 0.049 bits,
so each error a rule removes saves about 4.9 bits; naming one of 15 columns costs 3.9 bits,
so a rule with three tests (about 24 bits, discounted to 5) pays for itself if it removes two or three errors.


Now the ruleset. `build_ruleset(X, y)` takes the fitted tree and the training data and returns a
`C5RulesClassifier`, a scikit-learn classifier of its own that predicts with the rules. Fitting
`C5RulesClassifier()` directly on the data gives the same result in one step.

In [7]:
ruleset = tree.build_ruleset(X_train, y_train)
print(f"{len(ruleset.rules_)} rules in the ruleset (from {len(rules)} rules of the tree)")
print(f"test accuracy: tree {tree.score(X_test, y_test):.3f}   ruleset {ruleset.score(X_test, y_test):.3f}\n")
for line in ruleset.export_ruleset():
    print(line)

27 rules in the ruleset (from 39 rules of the tree)


test accuracy: tree 0.954   ruleset 0.961

Rule 1: IF total_day_minutes > 254 AND voice_mail_plan IN {no} AND total_eve_minutes > 183.75 AND total_night_minutes > 195.2 THEN churn  [cases=75, errors=0, confidence=0.987, lift=6.98]
Rule 2: IF total_day_minutes <= 243.95 AND total_day_minutes > 191 AND service_calls > 3.5 AND total_eve_minutes > 155.5 AND international_plan IN {no} THEN stays  [cases=63, errors=0, confidence=0.985, lift=1.15]
Rule 3: IF total_day_minutes <= 233.35 AND service_calls <= 3.5 AND international_plan IN {no} AND total_eve_minutes <= 272.05 AND total_eve_minutes > 259.3 AND total_night_minutes <= 223.95 THEN stays  [cases=60, errors=0, confidence=0.984, lift=1.15]
Rule 4: IF international_plan IN {yes} AND total_intl_minutes > 13.05 THEN churn  [cases=55, errors=0, confidence=0.982, lift=6.95]
Rule 5: IF international_plan IN {yes} AND total_intl_calls <= 2.5 THEN churn  [cases=52, errors=0, confidence=0.981, lift=6.94]
Rule 6: IF total_day_minutes > 220.75 AND

Fewer rules, each one shorter, and on this data the ruleset is also a little more accurate on the
test set than the tree it comes from (simplifying a rule often removes tests that only fitted
noise). The rules are sorted by confidence. `export_ruleset(as_frame=True)` gives the same table
as a DataFrame, which is handier for filtering:

In [8]:
table = ruleset.export_ruleset(as_frame=True)
table[table.prediction == "churn"]

,rule_id,conditions,prediction,cases,errors,confidence,lift
0,1,total_day_minutes > 254 AND voice_mail_plan IN...,churn,75.0,0.0,0.987,6.98
3,4,international_plan IN {yes} AND total_intl_min...,churn,55.0,0.0,0.982,6.95
4,5,international_plan IN {yes} AND total_intl_cal...,churn,52.0,0.0,0.981,6.94
6,7,total_day_minutes <= 138.45 AND service_calls ...,churn,43.0,0.0,0.978,6.91
7,8,total_day_minutes > 220.75 AND total_eve_minut...,churn,38.0,0.0,0.975,6.89
9,10,total_day_minutes <= 119.55 AND service_calls ...,churn,33.0,0.0,0.971,6.87
13,14,total_day_minutes > 233.35 AND total_eve_minut...,churn,48.0,1.0,0.960,6.79
16,17,total_day_minutes <= 160.3 AND service_calls >...,churn,78.0,3.0,0.950,6.72
17,18,total_day_minutes > 220.75 AND service_calls <...,churn,38.0,1.0,0.950,6.72
18,19,total_day_minutes <= 197.2 AND service_calls >...,churn,29.0,1.0,0.935,6.61


The last churn rules in the table have a confidence close to 0.5: on their own they would be weak.
They are kept because rules in a ruleset overlap and vote. A heavy user who also satisfies a
strong "stays" rule (for example, one with a voicemail plan) is outvoted towards "stays"; one who
satisfies no stronger rule is flagged, with a moderate confidence that says the call is close.

## 4. From rules to reasons, and from reasons to campaigns

Read the churn rules above and three stories appear, each pointing to a different action:

* **International plan.** Customers with the international plan who use a lot of international
  minutes, or who barely make international calls. Either the plan is expensive for heavy users
  or it is useless to light users. *Action: review the international offer for them.*
* **Service frustration.** Customers who called customer service four times or more, mostly
  with low or medium usage. *Action: a proactive call from a senior agent to solve the
  pending problem.*
* **Heavy users without a voicemail plan.** Very high day and evening minutes: these are the
  largest bills. *Action: a bundle or a loyalty discount.*

To use this, label every churn rule with its reason (here from the columns it tests; in practice
an analyst reads the rules and names them), and send each customer at risk the campaign of the
strongest rule they satisfy. `apply_ruleset(X)` does the second part: for every customer it gives
the prediction, the strongest rule (highest confidence) among those they satisfy that predict it,
and how many rules they satisfy in total.

In [9]:
def reason(rule_text):
    if "international_plan IN {yes}" in rule_text:
        return "International plan"
    if "service_calls >" in rule_text:
        return "Service frustration"
    if "total_day_minutes >" in rule_text:
        return "Heavy user, no voicemail plan"
    return "Other"

CAMPAIGN = {
    "International plan": "Review the international offer",
    "Service frustration": "Proactive call from a senior agent",
    "Heavy user, no voicemail plan": "Bundle or loyalty discount",
    "Other": "Generic retention offer",
}

assigned = ruleset.apply_ruleset(X_test)
assigned["actual"] = y_test
at_risk = assigned[assigned.prediction == "churn"].copy()
at_risk["reason"] = at_risk["rule"].map(reason)
at_risk["campaign"] = at_risk["reason"].map(CAMPAIGN)

plan = (at_risk.groupby(["reason", "campaign"])
        .agg(customers=("rule", "size"), actually_left=("actual", lambda s: round((s == "churn").mean(), 2)))
        .reset_index().sort_values("customers", ascending=False))
print(f"{len(at_risk)} test customers flagged as churn; base churn rate {np.mean(y_test == 'churn'):.2f}\n")
plan

197 test customers flagged as churn; base churn rate 0.14



,reason,campaign,customers,actually_left
0,"Heavy user, no voicemail plan",Bundle or loyalty discount,96,0.81
1,International plan,Review the international offer,55,1.00
2,Service frustration,Proactive call from a senior agent,46,0.91


Each segment is small enough to treat differently and large enough to matter, and in each one
the share of customers who actually left is several times the base rate. This is the "double
click" a black-box score does not give: the list of customers at risk comes already split by
reason, with the action that fits each reason.

For a single customer, the drill-down is one row:

In [10]:
for i in at_risk.index[:3]:
    row = X_test.loc[i]
    r = at_risk.loc[i]
    print(f"Customer {i}: plan intl={row.international_plan}, voicemail={row.voice_mail_plan}, "
          f"day minutes={row.total_day_minutes:.0f}, service calls={row.service_calls}")
    print(f"   rule {r.rule_id}: {r.rule}")
    print(f"   reason: {r.reason}   campaign: {r.campaign}   (satisfies {r.n_rules} rule(s))\n")

Customer 1102: plan intl=no, voicemail=no, day minutes=177, service calls=4
   rule 19: total_day_minutes <= 197.2 AND service_calls > 3.5 AND total_eve_minutes <= 155.5
   reason: Service frustration   campaign: Proactive call from a senior agent   (satisfies 3 rule(s))

Customer 4228: plan intl=no, voicemail=no, day minutes=264, service calls=0
   rule 1: total_day_minutes > 254 AND voice_mail_plan IN {no} AND total_eve_minutes > 183.75 AND total_night_minutes > 195.2
   reason: Heavy user, no voicemail plan   campaign: Bundle or loyalty discount   (satisfies 2 rule(s))

Customer 4366: plan intl=yes, voicemail=no, day minutes=160, service calls=4
   rule 17: total_day_minutes <= 160.3 AND service_calls > 3.5 AND total_eve_minutes <= 233.3
   reason: Service frustration   campaign: Proactive call from a senior agent   (satisfies 5 rule(s))



**The same exercise with the rules of the tree.** It works too, because every customer follows
exactly one rule of the tree. The difference is the length of the segments: the rules of the tree
carry every test of their path, many of them only there to separate other customers higher up,
while the ruleset's rules keep only the tests that matter, so each segment can be described in a
sentence. The ruleset is the version you would show to a marketing team.

In [11]:
tree_assigned = tree.apply_rules(X_test)
tree_churn = tree_assigned[tree_assigned.prediction == "churn"]
print(f"tree:    {tree_churn.rule_id.nunique()} distinct churn rules used in the test set, "
      f"{np.mean([r.count(' AND ') + 1 for r in tree_churn.rule]):.1f} tests per rule on average")
print(f"ruleset: {at_risk.rule_id.nunique()} distinct churn rules used in the test set, "
      f"{np.mean([r.count(' AND ') + 1 for r in at_risk.rule]):.1f} tests per rule on average")

tree:    17 distinct churn rules used in the test set, 5.0 tests per rule on average
ruleset: 15 distinct churn rules used in the test set, 2.9 tests per rule on average


## 5. From rules to the customer database

A campaign is usually run from a database or a CRM, not from a notebook. Because the ruleset is
only rules, it can be written as one SQL query with `to_sql`: every rule adds its vote to its
class, the class with most votes wins, and customers that no rule covers get the default class,
exactly as `predict` does. The query can be scheduled in the database to flag customers every
night, with no Python in production.

In [12]:
sql = ruleset.to_sql("customers")
print("\n".join(sql.splitlines()[:8]), "\n   ...")

-- C5.0 ruleset: rules vote with their confidence (x1000); ties go to the default class,
-- then to the first class; rows covered by no rule get the default class.
WITH votes AS (
  SELECT *,
    CASE WHEN "total_day_minutes" > 254 AND "voice_mail_plan" IN ('no') AND "total_eve_minutes" > 183.75 AND "total_night_minutes" > 195.2 THEN 987 ELSE 0 END
      + CASE WHEN "international_plan" IN ('yes') AND "total_intl_minutes" > 13.05 THEN 982 ELSE 0 END
      + CASE WHEN "international_plan" IN ('yes') AND "total_intl_calls" <= 2.5 THEN 981 ELSE 0 END
      + CASE WHEN "total_day_minutes" <= 138.45 AND "service_calls" > 3.5 AND "total_intl_calls" <= 5.5 THEN 978 ELSE 0 END 
   ...


A check with SQLite (any SQL database works the same way): load the test customers into a table,
run the query, and compare its predictions with the model's.

In [13]:
import sqlite3
con = sqlite3.connect(":memory:")
X_test.astype({c: object for c in ["international_plan", "voice_mail_plan", "state", "area_code"]}).to_sql("customers", con, index=False)
from_sql = pd.read_sql(sql, con)["prediction"].to_numpy()
print(f"SQL and ruleset.predict agree on {np.mean(from_sql == ruleset.predict(X_test)):.0%} of the {len(X_test)} test customers")

SQL and ruleset.predict agree on 100% of the 1500 test customers


## 6. A word of caution: rules describe, campaigns must be tested

A rule says *who* leaves and what they look like. It does not say that changing what the rule
tests will keep them: "service calls > 3" is a symptom of a problem, not a lever, and the
customers who call a lot may differ from the others in ways the data does not record. Before
rolling out a campaign to a whole segment, send it to a random part of the segment and keep the
rest as a control group; the difference in churn between the two is the effect of the campaign.
The rules choose where to look and what to try; the experiment tells whether it works.

## Summary

* `C5Classifier.export_rules()` and `apply_rules()` give the **rules of the tree**: one per leaf,
  mutually exclusive, every customer follows exactly one.
* `C5RulesClassifier` (or `tree.build_ruleset(X, y)`) gives a **ruleset** in the style of C5.0:
  rules simplified and selected by description length, with confidence and lift, that vote when
  they overlap, and a default class. It is a scikit-learn classifier of its own (`fit`, `predict`, `predict_proba`,
  pipelines, `GridSearchCV`).
* `apply_ruleset()` assigns each customer the strongest rule they satisfy, which is what you
  need to split a list of customers at risk by reason and campaign.
* `to_sql()` writes the ruleset as a SQL query that reproduces its predictions, to run the
  campaign from the customer database.

**Reference.** Quinlan, J. R. (1993). *C4.5: Programs for Machine Learning*. Morgan Kaufmann;
chapter 5 (from trees to rules). C5.0 refines that method; c50py's rulesets are checked against
the R package C50 in the repository's benchmark.